In [0]:
from pyspark.sql.functions import col, lag, round, current_timestamp
from pyspark.sql.window import Window

df_silver_pop = spark.read.table("silver.worldbank_population")
df_silver_countries = spark.read.table("silver.worldbank_countries")

df_gold_joined = df_silver_pop.join(
    df_silver_countries,
    df_silver_pop.country_iso3 == df_silver_countries.country_code,
    "inner"
).select(
    df_silver_countries.country_name,
    df_silver_countries.region_name,
    df_silver_countries.income_level,
    df_silver_pop.year,
    df_silver_pop.population,
    df_silver_countries.country_code,
    df_silver_countries.latitude,
    df_silver_countries.longitude
)

window_spec = Window.partitionBy("country_name").orderBy("year")

df_gold_analytics = (
    df_gold_joined
    .withColumn("prev_year_population", lag("population", 1).over(window_spec))
    .withColumn(
        "yoy_growth_percentage", 
        round(((col("population") - col("prev_year_population")) / col("prev_year_population")) * 100, 2)
    )
    .withColumn("gold_processed_at", current_timestamp())
)

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

(
    df_gold_analytics.write
    .format("delta")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("gold.population_analytics")
)

display(spark.sql("SELECT * FROM gold.population_analytics WHERE country_name = 'Brazil' ORDER BY year DESC LIMIT 10"))